# Family Relationships Ontology

This example is a part of [AI for Beginners Curriculum](http://github.com/microsoft/ai-for-beginners), and it has been inspired by [this blog post](https://habr.com/post/270857/).

I always find it difficult to remember different relationships between people in a family. In this example, we will take an ontology that defines family relationships, and the actual genealogical tree, and show how we can then perform automatic inference to find all relatives.

### Getting the Genealogical Tree

As an example, we will take genealogical tree of [Romanov Tsar Family](https://en.wikipedia.org/wiki/House_of_Romanov). The most common format for describing family relationships is [GEDCOM](https://en.wikipedia.org/wiki/GEDCOM). We will take Romanov family tree in GEDCOM format:

In [29]:
import csv
from collections import defaultdict

INPUT_CSV = "./data/query.csv"
OUTPUT_GED = "house_of_saud.ged"

people = {}
families = {}
spouse_pairs = set()

def norm(text):
    if text is None:
        return ""
    return text.strip()

def get_person(pid, name=""):
    if pid not in people:
        people[pid] = {
            "name": name or pid,
            "father": "",
            "mother": "",
            "spouses": set(),
            "children": set(),
        }
    elif name and not people[pid]["name"]:
        people[pid]["name"] = name
    return people[pid]

with open(INPUT_CSV, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        person_id = norm(row["person"].split("/")[-1]) if row.get("person") else ""
        person_name = norm(row.get("personLabel", ""))
        father_id = norm(row["father"].split("/")[-1]) if row.get("father") else ""
        father_name = norm(row.get("fatherLabel", ""))
        mother_id = norm(row["mother"].split("/")[-1]) if row.get("mother") else ""
        mother_name = norm(row.get("motherLabel", ""))
        spouse_id = norm(row["spouse"].split("/")[-1]) if row.get("spouse") else ""
        spouse_name = norm(row.get("spouseLabel", ""))
        child_id = norm(row["child"].split("/")[-1]) if row.get("child") else ""
        child_name = norm(row.get("childLabel", ""))

        if not person_id:
            continue

        p = get_person(person_id, person_name)

        if father_id:
            p["father"] = father_id
            get_person(father_id, father_name)

        if mother_id:
            p["mother"] = mother_id
            get_person(mother_id, mother_name)

        if spouse_id:
            p["spouses"].add(spouse_id)
            get_person(spouse_id, spouse_name)
            pair = tuple(sorted([person_id, spouse_id]))
            spouse_pairs.add(pair)

        if child_id:
            p["children"].add(child_id)
            get_person(child_id, child_name)

# Build family records
family_index = 1

# Families from father+mother links
parent_groups = defaultdict(list)
for pid, pdata in people.items():
    key = (pdata["father"], pdata["mother"])
    if key != ("", ""):
        parent_groups[key].append(pid)

for (father, mother), kids in parent_groups.items():
    fam_id = f"F{family_index}"
    family_index += 1
    families[fam_id] = {
        "husb": father,
        "wife": mother,
        "chil": sorted(set(kids))
    }

# Extra families from spouse pairs without children
for a, b in sorted(spouse_pairs):
    exists = False
    for fam in families.values():
        if set([fam["husb"], fam["wife"]]) == set([a, b]):
            exists = True
            break
    if not exists:
        fam_id = f"F{family_index}"
        family_index += 1
        families[fam_id] = {
            "husb": a,
            "wife": b,
            "chil": []
        }

# Attach FAMC/FAMS links
fams_by_person = defaultdict(list)
famc_by_child = {}

for fam_id, fam in families.items():
    if fam["husb"]:
        fams_by_person[fam["husb"]].append(fam_id)
    if fam["wife"]:
        fams_by_person[fam["wife"]].append(fam_id)
    for child in fam["chil"]:
        famc_by_child[child] = fam_id

with open(OUTPUT_GED, "w", encoding="utf-8") as out:
    out.write("0 HEAD\n")
    out.write("1 SOUR Wikidata-to-GEDCOM\n")
    out.write("1 CHAR UTF-8\n")

    for pid, pdata in sorted(people.items()):
        out.write(f"0 @{pid}@ INDI\n")
        out.write(f"1 NAME {pdata['name']}\n")
        if pid in famc_by_child:
            out.write(f"1 FAMC @{famc_by_child[pid]}@\n")
        for fam_id in fams_by_person[pid]:
            out.write(f"1 FAMS @{fam_id}@\n")

    for fam_id, fam in sorted(families.items()):
        out.write(f"0 @{fam_id}@ FAM\n")
        if fam["husb"]:
            out.write(f"1 HUSB @{fam['husb']}@\n")
        if fam["wife"]:
            out.write(f"1 WIFE @{fam['wife']}@\n")
        for child in fam["chil"]:
            out.write(f"1 CHIL @{child}@\n")

    out.write("0 TRLR\n")

print(f"Created {OUTPUT_GED}")

Created house_of_saud.ged


In [30]:
!head -15 data/house_of_saud.ged

0 HEAD
1 SOUR Wikidata-to-GEDCOM
1 CHAR UTF-8
0 @Q100158628@ INDI
1 NAME Mohammed bin Khalid Al Saud
0 @Q100268543@ INDI
1 NAME Noura bint Mohammed Al Saud
1 FAMC @F128@
0 @Q100579253@ INDI
1 NAME Hessa bint Trad Al Shaalan
1 FAMS @F44@
0 @Q100728676@ INDI
1 NAME Al Jawhara bint Fahd Al Saud
1 FAMC @F30@
0 @Q102205480@ INDI


To use GEDCOM file, we can use `python-gedcom` library:

In [31]:
import sys
!{sys.executable} -m pip install python-gedcom

This library takes away some of the technical problems with file parsing, but it still gives us pretty low-level access to all individuals and families in the tree. Here is how we can parse the file, and show the list of all individuals:

In [34]:
from gedcom.parser import Parser
from gedcom.element.individual import IndividualElement
from gedcom.element.family import FamilyElement
g = Parser()
g.parse_file('data/house_of_saud.ged')

In [35]:
d = g.get_element_dictionary()
[ (k,v.get_name()) for k,v in d.items() if isinstance(v,IndividualElement)]

[('@Q100158628@', ('Mohammed bin Khalid Al Saud', '')),
 ('@Q100268543@', ('Noura bint Mohammed Al Saud', '')),
 ('@Q100579253@', ('Hessa bint Trad Al Shaalan', '')),
 ('@Q100728676@', ('Al Jawhara bint Fahd Al Saud', '')),
 ('@Q102205480@', ('Aida Fustuq', '')),
 ('@Q102371457@',
  ('Abdullah bin Saud bin Muhammad bin Abdulaziz Al Saud', '')),
 ('@Q104520507@', ('Faisal bin Bandar bin Sultan Al Saud', '')),
 ('@Q104781582@', ('Saad bin Faisal Al Saud', '')),
 ('@Q105492473@', ('Jiluwi bin Abdulaziz Al Saud', '')),
 ('@Q1059677@', ('Khalid bin Saud bin Abdulaziz Al Saud', '')),
 ('@Q106181708@', ('Qumash bint Abdulaziz Al Saud', '')),
 ('@Q106463171@', ('Noura bint Mohammed bin Saud Al Kabir', '')),
 ('@Q106829332@', ('Nouf bint Abdulaziz Al Saud', '')),
 ('@Q106830671@', ('Noura bint Sultan Al Saud', '')),
 ('@Q106855706@', ('Jawaher bint Majid Al Saud', '')),
 ('@Q106886010@', ('Maha bint Mishari Al Saud', '')),
 ('@Q106916742@', ('Dalal bint Saud Al Saud', '')),
 ('@Q106949529@', ('

Here is how we can get information about families. Note that is gives us a list of **identifiers**, and we need to convert them to names if we want more clarity:

In [36]:
d = g.get_element_dictionary()
[ (k,[x.get_value() for x in v.get_child_elements()]) for k,v in d.items() if isinstance(v,FamilyElement)]

[('@F1@',
  ['@Q6758324@', '@Q109257408@', '@Q12243047@', '@Q125492927@', '@Q4664684@']),
 ('@F10@',
  ['@Q244206@',
   '@Q106916742@',
   '@Q109298072@',
   '@Q110260194@',
   '@Q12197692@',
   '@Q12199305@',
   '@Q12203729@',
   '@Q12207802@',
   '@Q12210014@',
   '@Q12215931@',
   '@Q12216543@',
   '@Q12217829@',
   '@Q12217836@',
   '@Q12221822@',
   '@Q12222475@',
   '@Q12223497@',
   '@Q12223498@',
   '@Q12223752@',
   '@Q12223790@',
   '@Q12229375@',
   '@Q12231217@',
   '@Q12231661@',
   '@Q12238127@',
   '@Q12240141@',
   '@Q12242556@',
   '@Q12243174@',
   '@Q12243227@',
   '@Q12245053@',
   '@Q12245610@',
   '@Q12247734@',
   '@Q16117995@',
   '@Q16118239@',
   '@Q16118369@',
   '@Q16118602@',
   '@Q16119071@',
   '@Q16119531@',
   '@Q16122140@',
   '@Q16123896@',
   '@Q16126730@',
   '@Q16128191@',
   '@Q16128196@',
   '@Q16128401@',
   '@Q20386059@',
   '@Q20396215@',
   '@Q20425317@',
   '@Q22686307@',
   '@Q22688830@',
   '@Q22689055@',
   '@Q22690424@',
   '@Q22690860@'

### Getting Family Ontology

Next, let's have a look at [family ontology](https://raw.githubusercontent.com/blokhin/genealogical-trees/master/data/header.ttl) defined as a set of Semantic Web triplets. This ontology defines such relationships as `isUncleOf`, `isCousinOf`, and many others. All those relationships are defined in terms of basic predicates `isMotherOf`, `isFatherOf`, `isBrotherOf` and `isSisterOf`. We will use automatic reasoning to deduce all other relationships using the ontology.

Here is a sample definition of `isAuntOf` property, which is defined as a composition of `isSisterOf` and `isParentOf` (*Aunt is a sister of one's parent*).

```
fhkb:isAuntOf a owl:ObjectProperty ;
    rdfs:domain fhkb:Woman ;
    rdfs:range fhkb:Person ;
    owl:propertyChainAxiom ( fhkb:isSisterOf fhkb:isParentOf ) .
```

In [37]:
!head -20 data/onto.ttl

@prefix fhkb: <http://www.example.com/genealogy.owl#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xml: <http://www.w3.org/XML/1998/namespace> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://www.example.com/genealogy.owl#> a owl:Ontology .

fhkb:DomainEntity a owl:Class .

fhkb:Man a owl:Class ;
    owl:equivalentClass [ a owl:Class ;
            owl:intersectionOf ( fhkb:Person [ a owl:Restriction ;
                        owl:onProperty fhkb:hasSex ;
                        owl:someValuesFrom fhkb:Male ] ) ] .

fhkb:Woman a owl:Class ;
    owl:equivalentClass [ a owl:Class ;
            owl:intersectionOf ( fhkb:Person [ a owl:Restriction ;


### Constructing Ontology for Inference

For simplicity, we will create one ontology file that will include original rules from family ontology, and facts about individuals from our GEDCOM file. We will go through the GEDCOM file and extract information about families and individuals, and convert them to triplets.

In [38]:
!cp data/onto.ttl .

gedcom_dict = g.get_element_dictionary()
individuals, marriages = {}, {}

def term2id(el):
    return "i" + el.get_pointer().replace('@', '').lower()

out = open("onto.ttl","a")

for k, v in gedcom_dict.items():
    if isinstance(v,IndividualElement):
        children, siblings = set(), set()
        idx = term2id(v)

        title = v.get_name()[0] + " " + v.get_name()[1]
        title = title.replace('"', '').replace('[', '').replace(']', '').replace('(', '').replace(')', '').strip()

        own_families = g.get_families(v, 'FAMS')
        for fam in own_families:
            children |= set(term2id(i) for i in g.get_family_members(fam, "CHIL"))

        parent_families = g.get_families(v, 'FAMC')
        if len(parent_families):
            for member in g.get_family_members(parent_families[0], "CHIL"): # NB adoptive families i.e len(parent_families)>1 are not considered (TODO?)
                if member.get_pointer() == v.get_pointer():
                    continue
                siblings.add(term2id(member))

        if idx in individuals:
            children |= individuals[idx].get('children', set())
            siblings |= individuals[idx].get('siblings', set())
        individuals[idx] = {'sex': v.get_gender().lower(), 'children': children, 'siblings': siblings, 'title': title}

    elif isinstance(v,FamilyElement):
        wife, husb, children = None, None, set()
        children = set(term2id(i) for i in g.get_family_members(v, "CHIL"))

        try:
            wife = g.get_family_members(v, "WIFE")[0]
            wife = term2id(wife)
            if wife in individuals: individuals[wife]['children'] |= children
            else: individuals[wife] = {'children': children}
        except IndexError: pass
        try:
            husb = g.get_family_members(v, "HUSB")[0]
            husb = term2id(husb)
            if husb in individuals: individuals[husb]['children'] |= children
            else: individuals[husb] = {'children': children}
        except IndexError: pass

        if wife and husb: marriages[wife + husb] = (term2id(v), wife, husb)

for idx, val in individuals.items():
    added_terms = ''
    if val['sex'] == 'f':
        parent_predicate, sibl_predicate = "isMotherOf", "isSisterOf"
    else:
        parent_predicate, sibl_predicate = "isFatherOf", "isBrotherOf"
    if len(val['children']):
        added_terms += " ;\n    fhkb:" + parent_predicate + " " + ", ".join(["fhkb:" + i for i in val['children']])
    if len(val['siblings']):
        added_terms += " ;\n    fhkb:" + sibl_predicate + " " + ", ".join(["fhkb:" + i for i in val['siblings']])
    out.write("fhkb:%s a owl:NamedIndividual, owl:Thing%s ;\n    rdfs:label \"%s\" .\n" % (idx, added_terms, val['title']))

for k, v in marriages.items():
    out.write("fhkb:%s a owl:NamedIndividual, owl:Thing ;\n    fhkb:hasFemalePartner fhkb:%s ;\n    fhkb:hasMalePartner fhkb:%s .\n" % v)

out.write("[] a owl:AllDifferent ;\n    owl:distinctMembers (")
for idx in individuals.keys():
    out.write("    fhkb:" + idx)
for k, v in marriages.items():
    out.write("    fhkb:" + v[0])
out.write("    ) .")
out.close()

In [39]:
!tail onto.ttl

    fhkb:hasFemalePartner fhkb:iq205051 ;
    fhkb:hasMalePartner fhkb:iq6932834 .
fhkb:if93 a owl:NamedIndividual, owl:Thing ;
    fhkb:hasFemalePartner fhkb:iq25336703 ;
    fhkb:hasMalePartner fhkb:iq25336702 .
fhkb:if99 a owl:NamedIndividual, owl:Thing ;
    fhkb:hasFemalePartner fhkb:iq93119034 ;
    fhkb:hasMalePartner fhkb:iq5429531 .
[] a owl:AllDifferent ;
    owl:distinctMembers (    fhkb:iq100158628    fhkb:iq100268543    fhkb:iq100579253    fhkb:iq100728676    fhkb:iq102205480    fhkb:iq102371457    fhkb:iq104520507    fhkb:iq104781582    fhkb:iq105492473    fhkb:iq1059677    fhkb:iq106181708    fhkb:iq106463171    fhkb:iq106829332    fhkb:iq106830671    fhkb:iq106855706    fhkb:iq106886010    fhkb:iq106916742    fhkb:iq106949529    fhkb:iq107007381    fhkb:iq108113176    fhkb:iq108822039    fhkb:iq109257408    fhkb:iq109257410    fhkb:iq109258685    fhkb:iq109298072    fhkb:iq110260183    fhkb:iq110260194    fhkb:iq111842953    fhkb:iq111995522    fhkb:iq111995705    fhkb:

### Doing Inference 

Now we want to be able to use this ontology for inference and for querying. We will use [RDFLib](https://github.com/RDFLib), library for reading RDF Graph in different formats, querying it, etc. 

For logical inference, we will use [OWL-RL](https://github.com/RDFLib/OWL-RL) library, which allows us to build **Closure** of the RDF Graph, i.e. add all possible concepts and relations that can be inferred.

In [40]:
!{sys.executable} -m pip install rdflib
!{sys.executable} -m pip install git+https://github.com/RDFLib/OWL-RL.git

  Cloning https://github.com/RDFLib/OWL-RL.git to /private/var/folders/2y/r7fngq1j5v58cg15sq53tr280000gn/T/pip-req-build-sn9t5ncj
  Running command git clone --filter=blob:none --quiet https://github.com/RDFLib/OWL-RL.git /private/var/folders/2y/r7fngq1j5v58cg15sq53tr280000gn/T/pip-req-build-sn9t5ncj
  Resolved https://github.com/RDFLib/OWL-RL.git to commit d91df6e2f2ef09382a549c225898a0a92376a39a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Let's open the ontology file and see how many triplets it contains:

In [41]:
import rdflib
from owlrl import DeductiveClosure, OWLRL_Extension

g = rdflib.Graph()
g.parse("onto.ttl", format="turtle")

print("Triplets found:%d" % len(g))

Triplets found:7499


Now let's build the closure, and see how the number of triplets increase:

In [44]:
DeductiveClosure(OWLRL_Extension).expand(g)


### Querying for Relatives 

Now we can query the graph to see different relations between people. We can use **SPARQL** language together with `query` method. In our case, let's see all **uncles** in our family tree:

In [ ]:
qres = g.query(
    """SELECT DISTINCT ?rel_name
       WHERE {
          ?a fhkb:isFirstCousinOf ?b .
          ?a rdfs:label ?aname .
          ?b rdfs:label ?bname .
       }""")

for row in qres:
    print("%s if first cousin of %s" % row)

Feel free to experiment with different other family relations. For example, you can have a look at `isAncestorOf` relation, which recurrently defines all ancestors of a given person.

Finally, let's clean up!

In [20]:
!rm onto.ttl